# imports

In [ ]:
from statsmodels.tsa.stattools import adfuller, acf, pacf
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import xgboost
import lightgbm
import catboost
from sktime.forecasting.arima import AutoARIMA
from sktime.forecasting.fbprophet import Prophet
from sktime.forecasting.croston import Croston
from sktime.forecasting.theta import ThetaForecaster
import optuna
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import pandas as pd
from pathlib import Path
import pathlib
import math
import holidays
import json
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import os
import random
import torch
import warnings
import xgboost as xgb
import lightgbm as lgb
from neuralforecast import NeuralForecast
from neuralforecast.models import (
    LSTM, GRU, TiDE, TCN, NBEATSx, NHITS, TFT, TSMixerx, KAN
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
warnings.filterwarnings("ignore")


USE_GPU = torch.cuda.is_available()
GPU_DEVICE = "0"   # first GPU
print("GPU available:", USE_GPU)


lasso_model = Lasso(alpha=0.01, max_iter=5000)
xgb_model = xgboost.XGBRegressor(n_jobs=-1,device="cuda",tree_method="hist")
lgb_model = lightgbm.LGBMRegressor(verbose=-1,device_type="gpu")
cat_model = catboost.CatBoostRegressor(verbose=0,task_type="GPU",devices=GPU_DEVICE)
rf_model = RandomForestRegressor(n_jobs=-1)
gb_model = GradientBoostingRegressor()
et_model = ExtraTreesRegressor(n_jobs=-1)
ridge_model = Ridge(alpha=1.0)
elasticnet_model = ElasticNet(alpha=0.01, l1_ratio=0.2, max_iter=5000)
linear_model = LinearRegression()
mlp_model = MLPRegressor(hidden_layer_sizes=(200, 100, 50), activation='relu', solver='adam', max_iter=1000, random_state=42)
svr_model = SVR()
dt_model = DecisionTreeRegressor()
auto_arimax_model= AutoARIMA(n_jobs=-1)
prophet_model = Prophet()
croston_model=Croston()
theta_model=ThetaForecaster(deseasonalize=False)
lstm_model=LSTM(h=1)

def make_nf_dataframe(y, X=None, unique_id="series_1"):
    """
    Convert y/X into NeuralForecast long format:
    columns: unique_id, ds, y, <exog...>
    """
    df_nf = pd.DataFrame({
        "unique_id": unique_id,
        "ds": pd.to_datetime(y.index),
        "y": y.values
    })

    if X is not None:
        Xc = X.copy()
        Xc.index = pd.to_datetime(Xc.index)
        Xc = Xc.reset_index(drop=False)
        Xc = Xc.rename(columns={Xc.columns[0]: "ds"})
        df_nf = df_nf.merge(Xc, on="ds", how="left")

    return df_nf

def hpo_models(X_train,y_train,models,opt_trials):
    """
    This function recieves a dictionary of models and performs hpo in each and every one of them
    based on the training size

    Inputs:
    models is dicitonary with the models that are tackled.
    opt_trials are the iterations for the optimization

    Outputs: 
    parameters which is a dictionary with the hyperparameters at hand
    """
    parameters={}
    for opt_model in models.keys():
        if opt_model == "Lasso":
            def objective_Lasso(trial):
                sugg_alpha = trial.suggest_float("alpha", 1e-5, 1)
                sugg_fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])  # Include intercept or not
                sugg_selection = trial.suggest_categorical("selection", ["cyclic", "random"])  # Update strategy

                # Initialize MinMaxScaler
                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                # Scale the training data
                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                # Initialize model with suggested hyperparameters
                model = Lasso(alpha=sugg_alpha, fit_intercept=sugg_fit_intercept, selection=sugg_selection, max_iter=3000, random_state=42)

                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    # 🔹 Unscale predictions **before** calculating MSE
                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse = mean_squared_error(y_val_original, y_pred)  # Compute MSE in the original scale
                    mse_scores.append(mse)

                return np.mean(mse_scores)

            study_Lasso = optuna.create_study(direction="minimize")
            study_Lasso.optimize(objective_Lasso, n_trials=opt_trials)
            parameters[opt_model] = study_Lasso.best_params
        

        if opt_model=="XGBoost":
            def objective_XGB(trial):

                params = {
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
                    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                }  
                model=xgboost.XGBRegressor(**params,device="cuda",tree_method="hist")
                
                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    
                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)
            study_XGB=optuna.create_study(direction="minimize")
            study_XGB.optimize(objective_XGB,n_trials=opt_trials)
            parameters[opt_model]=study_XGB.best_params

        if opt_model=="LightGBM":
            def objective_LGBM(trial):

                params = {
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
                    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                } 
                model=lightgbm.LGBMRegressor(**params,verbose=-1,device_type="gpu")
                
                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    
                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)
            study_LGBM=optuna.create_study(direction="minimize")
            study_LGBM.optimize(objective_LGBM,n_trials=opt_trials)
            parameters[opt_model]=study_LGBM.best_params

        if opt_model=="CatBoost":
            def objective_CatBoost(trial):

                params = {
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3,log=True),  # Log scale for better tuning
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "depth": trial.suggest_int("depth", 3, 12),  # Tree complexity
                    "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10, log=True),  # L2 regularization
                    "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),  # Randomness in sampling
                    #"colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),  # only on cpu!
                } 
                model=catboost.CatBoostRegressor(early_stopping_rounds=50, **params,task_type="GPU",devices=GPU_DEVICE, verbose=0)
                
                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    
                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)
            study_CatBoost=optuna.create_study(direction="minimize")
            study_CatBoost.optimize(objective_CatBoost,n_trials=opt_trials)
            parameters[opt_model]=study_CatBoost.best_params

        if opt_model=="RandomForest":
            def objective_RF(trial):

                params = {
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),  # Number of trees
                    "max_depth": trial.suggest_int("max_depth", 3, 30),  # Tree depth
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),  # Minimum leaf size
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),  # Minimum samples per split
                    "max_features": trial.suggest_float("max_features", 0.3, 1.0),  # Feature fraction
                    "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),  # Whether to sample data with replacement
                }
                model=RandomForestRegressor(**params)
                
                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    
                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)
            study_RF=optuna.create_study(direction="minimize")
            study_RF.optimize(objective_RF,n_trials=opt_trials)
            parameters[opt_model]=study_RF.best_params

        if opt_model == "GradientBoosting":
            def objective_GBR(trial):
                params = {
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
                    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                    "max_features": trial.suggest_float("max_features", 0.3, 1.0),
                }
                model = GradientBoostingRegressor(**params)
                
                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    
                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)
            
            study_GBR = optuna.create_study(direction="minimize")
            study_GBR.optimize(objective_GBR, n_trials=opt_trials)
            parameters[opt_model] = study_GBR.best_params

        if opt_model == "ExtraTrees":
            def objective_ETR(trial):
                params = {
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "max_depth": trial.suggest_int("max_depth", 3, 30),
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
                    "max_features": trial.suggest_float("max_features", 0.3, 1.0),
                    "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
                }
                model = ExtraTreesRegressor(**params)
                
                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    
                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)
            
            study_ETR = optuna.create_study(direction="minimize")
            study_ETR.optimize(objective_ETR, n_trials=opt_trials)
            parameters[opt_model] = study_ETR.best_params

        if opt_model == "Ridge":
            def objective_Ridge(trial):
                params = {
                    "alpha": trial.suggest_float("alpha", 1e-5, 10, log=True),
                    "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False])
                }

                # Initialize MinMaxScaler
                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                # Scale the training data
                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                # Initialize model with suggested hyperparameters
                model = Ridge(**params)

                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    # 🔹 Unscale predictions **before** calculating MSE
                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse = mean_squared_error(y_val_original, y_pred)  # Compute MSE in the original scale
                    mse_scores.append(mse)

                return np.mean(mse_scores)

            study_Ridge = optuna.create_study(direction="minimize")
            study_Ridge.optimize(objective_Ridge, n_trials=opt_trials)
            parameters[opt_model] = study_Ridge.best_params

        if opt_model == "ElasticNet":
            def objective_ElasticNet(trial):
                params = {
                    "alpha": trial.suggest_float("alpha", 1e-5, 10.0, log=True),  # Regularization strength
                    "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),  # Balance between L1 (Lasso) and L2 (Ridge)
                    "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),  # Whether to fit the intercept
                }

                # Initialize MinMaxScaler
                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                # Scale the training data
                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                # Initialize ElasticNet model with suggested hyperparameters
                model = ElasticNet(**params)

                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    # 🔹 Unscale predictions and validation target before calculating MSE
                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse = mean_squared_error(y_val_original, y_pred)  # Compute MSE in the original scale
                    mse_scores.append(mse)

                return np.mean(mse_scores)

            study_ElasticNet = optuna.create_study(direction="minimize")
            study_ElasticNet.optimize(objective_ElasticNet, n_trials=opt_trials)
            parameters[opt_model] = study_ElasticNet.best_params

        if opt_model == "SVR":
            def objective_SVR(trial):
                params = {
                    "C": trial.suggest_float("C", 1e-3, 100.0, log=True),  # Regularization parameter
                    "epsilon": trial.suggest_float("epsilon", 1e-4, 1.0, log=True),  # Epsilon-tube within which no penalty is given
                    "kernel": trial.suggest_categorical("kernel", ["linear", "poly", "rbf", "sigmoid"]),  # Kernel type
                    "gamma": trial.suggest_categorical("gamma", ["scale", "auto"]),  # Kernel coefficient
                    "shrinking": trial.suggest_categorical("shrinking", [True, False]),  # Shrinking heuristic
                }

                # Handle "degree" only if kernel is "poly"
                if params["kernel"] == "poly":
                    params["degree"] = trial.suggest_int("degree", 2, 5)  # Polynomial kernel degree

                # Initialize MinMaxScaler
                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                # Scale the training data
                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                # Initialize SVR model with suggested hyperparameters
                model = SVR(**params)

                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    # 🔹 Unscale predictions and validation target before calculating MSE
                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse = mean_squared_error(y_val_original, y_pred)  # Compute MSE in the original scale
                    mse_scores.append(mse)

                return np.mean(mse_scores)

            study_SVR = optuna.create_study(direction="minimize")
            study_SVR.optimize(objective_SVR, n_trials=opt_trials)
            parameters[opt_model] = study_SVR.best_params


        if opt_model == "DecisionTree":
            def objective_DTR(trial):
                params = {
                    "max_depth": trial.suggest_int("max_depth", 3, 30),  # Maximum depth of the tree
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),  # Minimum samples to split
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),  # Minimum samples per leaf
                    "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),  # Feature selection
                    "splitter": trial.suggest_categorical("splitter", ["best", "random"]),  # Splitting strategy
                }

                model = DecisionTreeRegressor(**params)

                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)

                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)

            study_DTR = optuna.create_study(direction="minimize")
            study_DTR.optimize(objective_DTR, n_trials=opt_trials)
            parameters[opt_model] = study_DTR.best_params


        if opt_model == "MLP":
            def objective_MLP(trial):
                # Determine the number of layers (2 to 4)
                num_layers = trial.suggest_int("num_layers", 2, 4)

                # Define hidden layer sizes with different neuron counts per layer
                hidden_layer_sizes = tuple(
                    trial.suggest_int(f"layer_{i+1}", 50, 500, step=50) for i in range(num_layers)
                )

                params = {
                    "hidden_layer_sizes": hidden_layer_sizes,  # Variable hidden layer structure
                    "activation": trial.suggest_categorical("activation", ["identity", "logistic", "tanh", "relu"]),  # Activation function
                    "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),  # L2 regularization term
                    "learning_rate": trial.suggest_categorical("learning_rate", ["constant", "invscaling", "adaptive"]),  # Learning rate strategy
                    "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 1e-1, log=True),  # Initial learning rate
                    "batch_size": trial.suggest_categorical("batch_size", ["auto", 32, 64, 128]),  # Batch size
                }

                # Initialize MinMaxScaler
                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                # Scale the training data
                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                # Initialize MLP model with suggested hyperparameters
                model = MLPRegressor(**params, max_iter=300)

                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    # 🔹 Unscale predictions and validation target before calculating MSE
                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse = mean_squared_error(y_val_original, y_pred)  # Compute MSE in the original scale
                    mse_scores.append(mse)

                return np.mean(mse_scores)

            study_MLP = optuna.create_study(direction="minimize")
            study_MLP.optimize(objective_MLP, n_trials=opt_trials)

            best_params = study_MLP.best_params
            num_layers = best_params.pop("num_layers")

            # Extract hidden layer sizes from the best params
            hidden_layer_sizes = tuple(best_params.pop(f"layer_{i+1}") for i in range(num_layers))
            best_params["hidden_layer_sizes"] = hidden_layer_sizes  # Store correctly

            parameters[opt_model] = best_params  # Store the final result properly


        if opt_model == "Prophet":
            def objective_prophet_ext(trial):
                """
                Objective function for tuning Prophet hyperparameters with Optuna,
                including exogenous features (X).
                """

                # 1) Suggest Prophet hyperparameters via Optuna
                params = {
                    "seasonality_mode": trial.suggest_categorical("seasonality_mode", ["additive", "multiplicative"]),
                    "seasonality_prior_scale": trial.suggest_float("seasonality_prior_scale", 0.01, 10.0),
                    "changepoint_prior_scale": trial.suggest_float("changepoint_prior_scale", 0.001, 0.5),
                    "holidays_prior_scale": trial.suggest_float("holidays_prior_scale", 0.01, 10.0),
                }

                # 2) Create a time series splitter
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                # 3) Perform the time series CV
                for train_idx, val_idx in tscv.split(X_train):
                    y_train_fold = y_train.iloc[train_idx]
                    X_train_fold = X_train.iloc[train_idx]  # exogenous features
                    y_val_fold = y_train.iloc[val_idx]
                    X_val_fold = X_train.iloc[val_idx]      # exogenous features

                    # Instantiate Prophet with the hyperparameters
                    model = Prophet(**params)

                    model.fit(y_train_fold, X=X_train_fold)

                    # 5) Predict on the validation fold
                    # We create an fh that covers the length of the validation fold
                    fh_val = np.arange(1, len(y_val_fold) + 1)

                    # Prophet also takes exogenous data for the forecast step
                    y_pred_fold = model.predict(fh=fh_val, X=X_val_fold)

                    # 6) Compute MSE for this fold
                    mse_scores.append(mean_squared_error(y_val_fold, y_pred_fold))

                # 7) Return average MSE as the objective to minimize
                return np.mean(mse_scores)

            # Create and run the study
            study_prophet_ext = optuna.create_study(direction="minimize")
            study_prophet_ext.optimize(objective_prophet_ext, n_trials=opt_trials)  # example: 20 trials

            # Extract the best hyperparameters
            parameters[opt_model] = study_prophet_ext.best_params

        if opt_model == "Croston":
            def objective_croston(trial):
                """
                Objective function for tuning Croston hyperparameters with Optuna.
                """

                # 1) Suggest hyperparameters
                params = {
                    "smoothing": trial.suggest_float("smoothing", 0.01, 1.0)  # Smoothing factor (0.01 to 1)
                }

                # 2) Time series cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                # 3) Perform time series CV
                for train_idx, val_idx in tscv.split(y_train):
                    y_train_fold = y_train.iloc[train_idx]
                    y_val_fold = y_train.iloc[val_idx]

                    # 4) Train Croston model
                    model = Croston(**params)
                    model.fit(y_train_fold)

                    # 5) Predict
                    fh_val = np.arange(1, len(y_val_fold) + 1)
                    y_pred_fold = model.predict(fh=fh_val)

                    # 6) Compute MSE for this fold
                    mse_scores.append(mean_squared_error(y_val_fold, y_pred_fold))

                # 7) Return average MSE
                return np.mean(mse_scores)

            # 8) Run Optuna study
            study_croston = optuna.create_study(direction="minimize")
            study_croston.optimize(objective_croston, n_trials=opt_trials)

            # 9) Store best parameters
            parameters[opt_model] = study_croston.best_params

        if opt_model == "Theta":
            def objective_theta(trial):
                """
                Objective function for tuning ThetaForecaster hyperparameters with Optuna.
                """
                if (y_train < 0).any() or (y_train == 0).any():
                    deseasonalize_option = False  # Force additive seasonality
                else:
                    deseasonalize_option = trial.suggest_categorical("deseasonalize", [True, False])
                # 1) Suggest hyperparameters
                params = {
                    "initial_level": trial.suggest_float("initial_level", 0.01, 1.0),  # SES smoothing factor
                    "deseasonalize": deseasonalize_option,
                    "sp": trial.suggest_int("sp", 1, min(36, len(y_train)//2 ))  # 96 for daily seasonality in 15-min data
                }

                # 2) Time series cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                # 3) Perform time series CV
                for train_idx, val_idx in tscv.split(y_train):
                    y_train_fold = y_train.iloc[train_idx]
                    y_val_fold = y_train.iloc[val_idx]

                    # 🔹 Ensure frequency is set
                    if y_train_fold.index.freq is None:
                        y_train_fold.index.freq = pd.infer_freq(y_train_fold.index)

                    # 4) Train ThetaForecaster
                    model = ThetaForecaster(**params)
                    model.fit(y_train_fold)

                    # 5) Predict
                    fh_val = np.arange(1, len(y_val_fold) + 1)
                    y_pred_fold = model.predict(fh=fh_val)

                    # 6) Compute MSE for this fold
                    mse_scores.append(mean_squared_error(y_val_fold, y_pred_fold))

                # 7) Return average MSE
                return np.mean(mse_scores)

            # 8) Run Optuna study
            study_theta = optuna.create_study(direction="minimize")
            study_theta.optimize(objective_theta, n_trials=opt_trials)

            # 9) Store best parameters
            parameters[opt_model] = study_theta.best_params


        if opt_model == "LSTM":

            def objective_lstm(trial):
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    y_train_fold = y_train.iloc[train_idx].copy()
                    X_train_fold = X_train.iloc[train_idx].copy()
                    y_val_fold = y_train.iloc[val_idx].copy()
                    X_val_fold = X_train.iloc[val_idx].copy()

                    h = len(y_val_fold)
                    train_len = len(y_train_fold)

                    max_input_size = min(10 * h, train_len - 1)
                    if max_input_size < 1:
                        return float("inf")

                    params = {
                        "h": h,
                        "input_size": trial.suggest_int("input_size", h, max(h, max_input_size)),
                        "encoder_n_layers": trial.suggest_int("encoder_n_layers", 1, 4),
                        "encoder_hidden_size": trial.suggest_int("encoder_hidden_size", 32, 256, step=32),
                        "encoder_dropout": trial.suggest_float("encoder_dropout", 0.0, 0.5),
                        "decoder_hidden_size": trial.suggest_int("decoder_hidden_size", 32, 256, step=32),
                        "decoder_layers": trial.suggest_int("decoder_layers", 1, 3),
                        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
                        "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
                        "max_steps": 100,
                        "scaler_type": "minmax",
                        "random_seed": SEED,
                        "hist_exog_list": list(X_train.columns),   # your engineered lag features
                    }

                    try:
                        train_df_nf = make_nf_dataframe(y_train_fold, X_train_fold, unique_id="series_1")

                        model = LSTM(**params)
                        nf = NeuralForecast(models=[model], freq="15min")
                        nf.fit(df=train_df_nf)

                        # For pure historical exogs, predict does not need future exogenous df
                        preds = nf.predict()
                        pred_col = preds.columns.difference(["unique_id", "ds"])[0]

                        y_pred_fold = preds[pred_col].values

                        if len(y_pred_fold) != len(y_val_fold):
                            return float("inf")

                        if np.isnan(y_pred_fold).any() or np.isnan(y_val_fold.to_numpy()).any():
                            return float("inf")

                        mse_scores.append(mean_squared_error(y_val_fold.to_numpy(), y_pred_fold))

                    except Exception:
                        return float("inf")

                return float(np.mean(mse_scores))

            # 8) Run Optuna study
            study_lstm = optuna.create_study(direction="minimize")
            study_lstm.optimize(objective_lstm, n_trials=opt_trials)

            # 9) Store best parameters
            parameters[opt_model] = study_lstm.best_params
    return parameters

def find_best_arima_params(y, max_lag=20):
    """
    Automatically finds the best (p, d, q) parameters for an ARIMA model.
    
    Parameters:
    - y: time series data (Pandas Series)
    - max_lag: max lag to consider in ACF/PACF (default: 20)
    
    Returns:
    - (p, d, q): best parameters for ARIMA model
    """
    # Step 1: Determine differencing order (d) using ADF Test
    def adf_test(series):
        result = adfuller(series, autolag='AIC')
        return 0 if result[1] < 0.05 else 1  # 0 if stationary, otherwise 1
    
    d = adf_test(y)
    
    # Apply differencing if needed
    y_diff = y.diff(d).dropna() if d > 0 else y

    # Adjust max_lag to be at most 50% of the available data
    adjusted_max_lag = min(max_lag, len(y_diff) // 2)

    # Step 2: Determine AR order (p) using PACF
    pacf_values = pacf(y_diff, nlags=adjusted_max_lag)
    p = np.argmax(np.abs(pacf_values) < 1.96 / np.sqrt(len(y_diff)))  # First lag where PACF is insignificant

    # Step 3: Determine MA order (q) using ACF
    acf_values = acf(y_diff, nlags=adjusted_max_lag)
    q = np.argmax(np.abs(acf_values) < 1.96 / np.sqrt(len(y_diff)))  # First lag where ACF is insignificant

    return p, d, q


def forecasting_households(project_path,
                          df_original,
                          df_weather,
                          date,
                          forecast_end_date,
                          forecast_horizon,
                          training_size,
                          feature_selection,
                          plot_forecast,
                          hyperparameter_opt,
                          models,
                          number_of_houses,
                          opt_trials=30):
    """
    Inputs: 
    df_original=pd.Dataframe  
    df_weather=pd.Dataframe
    date=timestamp tha includes the first timestamp of the forecasting period
    forecast_end_date= the last timestamp of the forecasting period
    forecast_horizon= integer, the forecasting horizon
    training_size= integer, training size
    feature_selection= bool, if true you make feature selection
    plot_forecast=bool, if true you save a plot 
    hyperparameter_opt=bool, if true HPO is performed
    opt_trials=integer, number of HPO trials
    models=dicitonary, the models that the user is intrested on adding to the study
    """

    path = pathlib.Path(project_path) / "data"
    #df_original = pd.read_csv(path / "inputs" / f"random_hh_{number_of_houses}.csv", index_col=0,delimiter=";")
    #df_weather=pd.read_csv(path / "inputs" / "weather.csv", index_col=0, parse_dates=[0])
    df_weather.index = pd.to_datetime(df_weather.index, utc=True).tz_localize(None)
    df_original.index = pd.date_range(start="2024-01-01 00:00:00", periods=len(df_original), freq="15min")
    df_original.rename(columns={"0": "Transformer Power"},inplace=True)
    target="Transformer Power"

    df_original=df_original[[target]]

    #date="2024-01-11 00:00:00" #this date includes the first timestamp of the forecasting period! 
    #forecast_end_date="2024-01-12 00:00:00"
    #forecast_horizon=96
    #training_size=forecast_horizon*6 #forecast_horizon*5 # (forecast_horizon*2+2) minimum for pandas  # if HPO is true then all before+ (cv_fold*forecast_horizon+2)
    #feature_selection=True
    #plot_forecast=True
    #hyperparameter_opt=True
    #opt_trials=2

    initial_date = pd.Timestamp(date)  # Initial date
    end_date = pd.Timestamp(forecast_end_date)
    time_step = (df_original.index[1] - df_original.index[0]).seconds // 60  # Time difference in minutes
    step_size = pd.Timedelta(minutes=forecast_horizon * time_step)  # Each step is 15 minutes
    n_iterations = math.ceil((end_date - initial_date) / step_size)
    weather_feature_list= df_weather.columns.to_list()


    all_predictions = []  # Initialize an empty list to store predictions

    for i in range(n_iterations):


        print(i)
        current_date = initial_date + i * step_size
        df_trafo = df_original[df_original.index < current_date]
        df_trafo=df_trafo.tail(training_size)

        # Ensure timestamps are datetime
        df_trafo.index = pd.to_datetime(df_trafo.index)
        df_weather.index = pd.to_datetime(df_weather.index)

        # Define the start and end timestamps
        start_timestamp = df_trafo.index.min()
        end_timestamp = df_weather.index.max()

        # Merge using an outer join to include all timestamps
        df = pd.merge(df_trafo, df_weather, left_index=True, right_index=True, how="outer")

        # Filter to ensure the merged DataFrame includes only the desired time range
        df = df.loc[start_timestamp:end_timestamp]

        # Find the first occurrence of NaN in "MS_NS_Trafo_1"
        first_nan_index = df[df[target].isna()].index.min()

        # If no NaN is found, return an empty DataFrame
        if pd.isna(first_nan_index):
            filtered_df = pd.DataFrame()  # Empty DataFrame
        else:
            # Get the position of the first NaN index
            start_position = df.index.get_loc(first_nan_index)

            # Define the end position (96 steps ahead, ensuring it doesn't exceed DataFrame length)
            end_position = min(start_position + forecast_horizon, len(df))

            # Slice the DataFrame
            filtered_df = df.iloc[:end_position].copy()

        ############ feature creation 
        for lag in range(forecast_horizon, forecast_horizon*2 + 1):
            filtered_df[f"{target}_lag_{lag}"] = df[target].shift(lag)


        for lag in range(1, forecast_horizon + 1):
            for g in weather_feature_list:
                filtered_df[f"{g}_lag_{lag}"] = df[g].shift(lag)

        # Ensure the index is datetime
        filtered_df.index = pd.to_datetime(filtered_df.index)

        # Extract basic time features
        filtered_df['minute'] = filtered_df.index.minute
        filtered_df['hour'] = filtered_df.index.hour
        filtered_df['day_of_week'] = filtered_df.index.dayofweek
        filtered_df['day_of_year'] = filtered_df.index.dayofyear
        filtered_df['week'] = filtered_df.index.isocalendar().week
        filtered_df['month'] = filtered_df.index.month
        filtered_df['year'] = filtered_df.index.year
        filtered_df['is_weekend'] = (filtered_df['day_of_week'] >= 5).astype(int)  # 1 if Saturday/Sunday, else 0

        # Add German holidays
        german_holidays = holidays.Germany()
        filtered_df['holiday'] = filtered_df.index.to_series().apply(lambda x: 1 if x in german_holidays else 0)

        # Define periods for cyclical encoding
        minute_period = 60
        hour_period = 24
        week_period = 7
        month_period = 12
        year_period = 365.25  # Accounting for leap years

        # Apply sine and cosine transformations for cyclical features
        filtered_df['minute_sin'] = np.sin(2 * np.pi * filtered_df['minute'] / minute_period)
        filtered_df['minute_cos'] = np.cos(2 * np.pi * filtered_df['minute'] / minute_period)
        filtered_df['hour_sin'] = np.sin(2 * np.pi * filtered_df['hour'] / hour_period)
        filtered_df['hour_cos'] = np.cos(2 * np.pi * filtered_df['hour'] / hour_period)
        filtered_df['dayofweek_sin'] = np.sin(2 * np.pi * filtered_df['day_of_week'] / week_period)
        filtered_df['dayofweek_cos'] = np.cos(2 * np.pi * filtered_df['day_of_week'] / week_period)
        filtered_df['dayofyear_sin'] = np.sin(2 * np.pi * filtered_df['day_of_year'] / year_period)
        filtered_df['dayofyear_cos'] = np.cos(2 * np.pi * filtered_df['day_of_year'] / year_period)
        filtered_df['week_sin'] = np.sin(2 * np.pi * filtered_df['week'] / week_period)
        filtered_df['week_cos'] = np.cos(2 * np.pi * filtered_df['week'] / week_period)
        filtered_df['month_sin'] = np.sin(2 * np.pi * filtered_df['month'] / month_period)
        filtered_df['month_cos'] = np.cos(2 * np.pi * filtered_df['month'] / month_period)
        filtered_df = filtered_df.dropna(subset=[col for col in filtered_df.columns if col != target])
        # Define the split point (last forecast_horizon rows)
        split_point = len(filtered_df) - forecast_horizon

        # Ensure there's enough data to split
        if split_point > 0:
            # Split y (target variable)
            y_train = filtered_df[target].iloc[:split_point]
            y_test = filtered_df[target].iloc[split_point:]

            # Split X (features) - exclude target
            X_train = filtered_df.drop(columns=[target]).iloc[:split_point]
            X_test = filtered_df.drop(columns=[target]).iloc[split_point:]
        else:
            raise ValueError("Not enough data to create a training and prediction split!")
        
        ######## feature selection
        if feature_selection==True:

            correlations = X_train.corrwith(y_train)

            # Select features with absolute correlation >= 0.2
            selected_features = correlations[abs(correlations) >= 0.1].index

            # Filter X_train to keep only the selected features
            X_train = X_train[selected_features]
            X_test=X_test[selected_features]

        ####### HPO HERE
        if hyperparameter_opt==True:
            opt_parameters={}
            opt_parameters=hpo_models(X_train,y_train,models,opt_trials)
        else:
            opt_parameters={
                'Lasso': {'alpha': 1.00},
                'XGBoost': {'n_estimators': 100,},
                'LightGBM': {'n_estimators': 100,},
                'CatBoost': {'n_estimators': 100},
                'RandomForest': {'n_estimators': 100,},
                'GradientBoosting': {'n_estimators': 100},
                'ExtraTrees': {'n_estimators': 100,},
                'Ridge': {'alpha': 1.0},
                'ElasticNet': {'alpha': 1.0},
                'MLP': {'alpha': 0.0001},
                'SVR': {'kernel': 'rbf',},
                'DecisionTree': {'criterion': 'squared_error'},
                'Prophet': {'seasonality_mode': 'additive'},
                'LSTM': {
                        'input_size': forecast_horizon,
                        'encoder_n_layers': 2,
                        'encoder_hidden_size': 128,
                        'encoder_dropout': 0.0,
                        'context_size': 10,
                        'decoder_hidden_size': 128,
                        'decoder_layers': 1,
                        'learning_rate': 1e-3,
                        'batch_size': 32
                    },
                'Croston':{'smoothing':0.1},
                'Theta':{'deseasonalize':True},
                }
        # Update models with optimized parameters
        # Dynamically recreate only the selected models using optimized parameters
        reinitialized_models = {}

        for name, params in opt_parameters.items():
            if name in models:  # Ensure only user-selected models are reinitialized
                if name == "Lasso":
                    reinitialized_models[name] = Lasso(**params, max_iter=5000)
                elif name == "XGBoost":
                    reinitialized_models[name] = xgboost.XGBRegressor(**params,device="cuda",tree_method="hist",random_state=SEED)
                elif name == "LightGBM":
                    reinitialized_models[name] = lightgbm.LGBMRegressor(**params,device_type="gpu", verbose=-1,random_state=SEED)
                elif name == "CatBoost":
                    reinitialized_models[name] = catboost.CatBoostRegressor(**params, task_type="GPU",devices=GPU_DEVICE,verbose=0, random_seed=SEED)
                elif name == "RandomForest":
                    reinitialized_models[name] = RandomForestRegressor(**params,n_jobs=-1, random_state=SEED)
                elif name == "GradientBoosting":
                    reinitialized_models[name] = GradientBoostingRegressor(**params, random_state=SEED)
                elif name == "ExtraTrees":
                    reinitialized_models[name] = ExtraTreesRegressor(**params, n_jobs=-1,random_state=SEED)
                elif name == "Ridge":
                    reinitialized_models[name] = Ridge(**params)
                elif name == "ElasticNet":
                    reinitialized_models[name] = ElasticNet(**params, max_iter=5000)
                elif name == "DecisionTree":
                    reinitialized_models[name] = DecisionTreeRegressor(**params)
                elif name == "MLP":
                    reinitialized_models[name] = MLPRegressor(**params, max_iter=1000, random_state=SEED)
                elif name == "SVR":
                    reinitialized_models[name] = SVR(**params)
                elif name == "Prophet":
                    reinitialized_models[name] = Prophet(**params)
                elif name == "Croston":
                    reinitialized_models[name] = Croston(**params)
                elif name == "Theta":
                    reinitialized_models[name] = ThetaForecaster(**params)
                elif name == "LSTM":
                    reinitialized_models[name] = LSTM(
                            h=forecast_horizon,
                            input_size=params.get("input_size", forecast_horizon),
                            encoder_n_layers=params.get("encoder_n_layers", 2),
                            encoder_hidden_size=params.get("encoder_hidden_size", 128),
                            encoder_dropout=params.get("encoder_dropout", 0.0),
                            context_size=params.get("context_size", 10),
                            decoder_hidden_size=params.get("decoder_hidden_size", 128),
                            decoder_layers=params.get("decoder_layers", 1),
                            learning_rate=params.get("learning_rate", 1e-3),
                            batch_size=params.get("batch_size", 32),
                            max_steps=100,
                            scaler_type="minmax",
                            random_seed=SEED,
                            hist_exog_list=list(X_train.columns)
                        )
        # Replace old models with the newly initialized ones
        models = reinitialized_models


        #find best params for autoArima
        suggested_p, suggested_d, suggested_q= find_best_arima_params(y_train, forecast_horizon)

        # Define scalers
        scaler = MinMaxScaler()
        scaled_target = MinMaxScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        y_train_scaled = scaled_target.fit_transform(y_train.values.reshape(-1, 1)).flatten()

        y_preds = {}

        # Train the models
        for name, model in models.items():
            print(f"Training {name}...")
            if name in ["Lasso", "Ridge", "ElasticNet", "LinearRegression", "MLP", "SVR"]:
                model.fit(X_train_scaled, y_train_scaled)
            elif name in ["AutoARIMAX","Prophet"]:
                model.fit(y_train,X=X_train)
            elif name in ["Croston","Theta"]:
                # Ensure y_train has a valid frequency before passing to ThetaForecaster
                if y_train.index.freq is None:
                    inferred_freq = pd.infer_freq(y_train.index)
                    
                    if inferred_freq is None:
                        raise ValueError("Cannot infer a valid frequency for y_train. Check for missing or irregular timestamps.")
                    
                    y_train = y_train.asfreq(inferred_freq)  # Set the inferred frequency

                model.fit(y_train)

            elif name in ["LSTM"]:
                train_df_nf = make_nf_dataframe(y_train, X_train, unique_id="series_1")
                model = NeuralForecast(models=[model], freq="15min")
                model.fit(df=train_df_nf)
                models[name] = model

            else:
                model.fit(X_train, y_train)
            print(f"{name} training complete.")

        # Make predictions
        for name, model in models.items():
            if name in ["Lasso", "Ridge", "ElasticNet", "LinearRegression", "MLP", "SVR"]:
                y_preds[name] = scaled_target.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1)).flatten()
            elif name in ["AutoARIMAX","Prophet"]:
                fh = np.arange(1, len(y_test) + 1)
                y_pred_auto = model.predict(fh=fh, X=X_test)
                y_pred_auto.index = y_test.index
                y_preds[name] = y_pred_auto.values
            elif name in ["Croston","Theta"]:
                fh = np.arange(1, len(y_test) + 1)
                y_pred_univariate=model.predict(fh=fh)
                y_pred_univariate.index = y_test.index
                y_preds[name] = y_pred_univariate.values

            elif name in ["LSTM"]:
                preds_nf = model.predict()
                pred_col = preds_nf.columns.difference(["unique_id", "ds"])[0]
                preds_nf = preds_nf.copy()
                preds_nf["ds"] = pd.to_datetime(preds_nf["ds"])
                preds_nf = preds_nf.set_index("ds")

                y_pred_lstm = preds_nf.loc[y_test.index, pred_col]
                y_preds[name] = y_pred_lstm.values
            else:
                y_preds[name] = model.predict(X_test)


        y_pred_df = pd.DataFrame(y_preds, index=y_test.index)

        # Extract the actual values from df_original
        y_actual = df_original.loc[y_test.index, target]

        inferred_freq = pd.infer_freq(df_original.index)
        # Convert frequency to a timedelta object
        if inferred_freq:
            freq_timedelta = pd.to_timedelta(inferred_freq)
            freq_minutes = freq_timedelta.total_seconds() / 60  # Convert to minutes
        else:
            raise ValueError("Could not infer frequency from the dataset index.")
        # Compute MSE for each model
        time_offset = pd.Timedelta(minutes=freq_minutes * 96) #instead of 96 put forecast_horizon for naive comparion of prediction horizon

        y_pred_naive = df_original.loc[y_test.index - time_offset, target]


        mse_scores = {name: mean_squared_error(y_actual, y_preds[name]) for name in models.keys()}

        mse_scores["Naive Forecast"] = mean_squared_error(y_actual, y_pred_naive)

        # Print MSE results
        for name, mse in mse_scores.items():
            print(f"MSE - {name}: {mse:.5f}")

        y_pred_df["Naive Forecast"] = y_pred_naive.values
        y_pred_df["Actual"]= y_actual.values
        all_predictions.append(y_pred_df)

    ###### save predictions
    final_predictions = pd.concat(all_predictions)
    # Define the output path
    path = pathlib.Path(project_path) / "data" / "outputs"
    path.mkdir(parents=True, exist_ok=True)  # Ensure the directory exists

    # Define the filename
    filename = f"final_predictions_{number_of_houses}_houses and {forecast_horizon} steps.csv"
    filepath = path / filename  # Full path

    # Save as CSV with index
    final_predictions.to_csv(filepath, index=True)

    print(f"CSV saved at: {filepath}")

    ###### save hyperparameters
    # Define the output path
    path = pathlib.Path(project_path) / "data" / "outputs"
    path.mkdir(parents=True, exist_ok=True)  # Ensure the directory exists
    # Define the filename
    filename = f"opt_parameters for {number_of_houses}nr. houses and {forecast_horizon} steps.json"
    filepath = path / filename  # Full path

    # Save the dictionary as a JSON file
    with open(filepath, "w") as f:
        json.dump(opt_parameters, f, indent=4)

    print(f"JSON saved at: {filepath}")

    if plot_forecast==True:

        # Define the path
        path = pathlib.Path(project_path) / "data" / "plots"
        path.mkdir(parents=True, exist_ok=True)  # Ensure the directory exists

        # Define the filename
        filename = f"actual vs predictions for {number_of_houses} nr. houses and {forecast_horizon} steps.png"
        filepath = path / filename  # Full path

        plt.figure(figsize=(12, 6))

        # Get a colormap and a list of bright colors
        colors = list(mcolors.TABLEAU_COLORS.values())  # Avoid dark colors

        # Determine the first timestamp in final_predictions
        first_timestamp = final_predictions.index.min()
        previous_day_start = first_timestamp - pd.Timedelta(days=1)  # Get one day before

        # Extract the previous day's actual values
        previous_day_actual = df_original.loc[previous_day_start:first_timestamp, "Transformer Power"]

        # Extract current actual values from final_predictions
        current_actual = final_predictions["Actual"]

        # Concatenate both to create a single continuous line
        actual_combined = pd.concat([previous_day_actual, current_actual])

        # Plot the single continuous black line for actual values
        plt.plot(actual_combined.index, actual_combined, label="Actual", color='black', linewidth=2)

        # Plot each forecast column
        for i, column in enumerate(final_predictions.columns):
            if column != "Actual":  # Skip "Actual" since it's already plotted
                plt.plot(final_predictions.index, final_predictions[column], label=column, color=colors[i % len(colors)])

        # Add legend with a white background and fixed position
        plt.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='black')

        # Labels and title
        plt.xlabel("Time")
        plt.ylabel("Value")
        plt.title("Forecast vs Actual")

        # Save the plot
        plt.savefig(filepath, dpi=300, bbox_inches='tight')  # High-quality save
        plt.close()  # Close the figure to free memory

        print(f"Plot saved at: {filepath}")



Seed set to 1


GPU available: True


# start

In [ ]:
models = {
"Lasso": lasso_model,
"XGBoost": xgb_model,
"LightGBM": lgb_model,
"CatBoost": cat_model,
"RandomForest": rf_model,
"GradientBoosting": gb_model,
"ExtraTrees": et_model,
"Ridge": ridge_model,
"ElasticNet": elasticnet_model,
"LinearRegression": linear_model,
"MLP": mlp_model,
"SVR": svr_model,
"DecisionTree": dt_model,
"LSTM": lstm_model,
}

project_path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone"
path = pathlib.Path(project_path) / "DataCleaning" / "clean"

first_set_of_house = 10
max_number_of_houses = 10

df_weather=pd.read_csv(path  / "weather.csv", index_col=0, parse_dates=[0])

date="2024-01-25 00:00:00" #this date includes the first timestamp of the forecasting period! 
forecast_end_date="2024-01-26 00:00:00"
forecast_horizon=96
training_size=96*7*3*2 #forecast_horizon*5 # (forecast_horizon*2+2) minimum for pandas  # if HPO is true then all before+ (cv_fold*forecast_horizon+2)
feature_selection=True
plot_forecast=True
hyperparameter_opt=True
opt_trials=30

for number_of_houses in range(first_set_of_house,
                            max_number_of_houses+1, 
                            10):


    df_original = pd.read_csv(path / f"net_load_{number_of_houses}_buildings.csv", index_col=0,delimiter=",")
    forecasting_households(project_path,
                          df_original,
                          df_weather,
                          date,
                          forecast_end_date,
                          forecast_horizon,
                          training_size,
                          feature_selection,
                          plot_forecast,
                          hyperparameter_opt,
                          models,
                          number_of_houses,
                          opt_trials)
    print("the amount of houses that we just did was", number_of_houses)

0


[I 2026-03-12 19:33:45,519] A new study created in memory with name: no-name-0120ae49-2e66-4aa8-8134-9983e40e356e
[I 2026-03-12 19:33:45,557] Trial 0 finished with value: 2.0539232619011033e-05 and parameters: {'alpha': 0.19631989176282935, 'fit_intercept': True, 'selection': 'cyclic'}. Best is trial 0 with value: 2.0539232619011033e-05.
[I 2026-03-12 19:33:45,593] Trial 1 finished with value: 2.0539232619011033e-05 and parameters: {'alpha': 0.41203540180238496, 'fit_intercept': True, 'selection': 'random'}. Best is trial 0 with value: 2.0539232619011033e-05.
[I 2026-03-12 19:33:45,631] Trial 2 finished with value: 0.000268087242730269 and parameters: {'alpha': 0.8135132218247257, 'fit_intercept': False, 'selection': 'random'}. Best is trial 0 with value: 2.0539232619011033e-05.
[I 2026-03-12 19:33:46,060] Trial 3 finished with value: 0.00018034540984555208 and parameters: {'alpha': 0.3257987416257844, 'fit_intercept': False, 'selection': 'cyclic'}. Best is trial 0 with value: 2.053923

┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ hist_encoder │ LSTM          │  186 K │ train │     0 │
│ 4 │ mlp_decoder  │ MLP           │  4.2 K │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 190 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 190 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 10                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

`Trainer.fit` stopped: `max_steps=1000` reached.


LSTM training complete.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

MSE - Lasso: 0.00001
MSE - XGBoost: 0.00001
MSE - LightGBM: 0.00001
MSE - CatBoost: 0.00001
MSE - RandomForest: 0.00001
MSE - GradientBoosting: 0.00001
MSE - ExtraTrees: 0.00001
MSE - Ridge: 0.00001
MSE - ElasticNet: 0.00001
MSE - MLP: 0.00001
MSE - SVR: 0.00001
MSE - DecisionTree: 0.00001
MSE - LSTM: 0.00001
MSE - Naive Forecast: 0.00001
CSV saved at: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\data\outputs\final_predictions_10_houses and 96 steps.csv
JSON saved at: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\data\outputs\opt_parameters for 10nr. houses and 96 steps.json
Plot saved at: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\data\plots\actual vs predictions for 10 nr. houses and 96 steps.png
the amount of houses that we just did was 10


# end

for 1 home is 218 minutes so 3.5 hours

Germany 28 homes *5 
Ireland 20 homes *5
Portugal 22 homes *5 = 350

350*3.5= 1225 hours = 50 days